Libraries

In [34]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from typing import Dict, List, Optional, Tuple, Union
from torch.utils.data import Dataset, DataLoader
from typing import Optional, Tuple, List
import torch.nn.functional as F
from torch.optim import AdamW
from pathlib import Path
import torch.nn as nn
from PIL import Image
import numpy as np
import open_clip
import argparse
import logging
import shutil
import torch
import tqdm
import math
import json
import glob
import time
import math
import os
import re

In [2]:
try:
    import nibabel as nib
    HAS_NIBABEL = True
except ImportError:
    HAS_NIBABEL = False

In [3]:
try:
    import open_clip
    HAS_OPENCLIP = True
except ImportError:
    HAS_OPENCLIP = False

In [4]:
try:
    from transformers import AutoTokenizer, CLIPModel, CLIPProcessor
    HAS_HF = True
except ImportError:
    HAS_HF = False

In [5]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)
log = logging.getLogger(__name__)

DHN-NCE Loss (Decoupled Hard Negative Noise Contrastive Estimation Loss)

In [6]:
class DHNNCELoss(nn.Module):
    def __init__(self, tau: float = 0.6, beta1: float = 0.15, beta2: float = 0.15):
        super().__init__()
        self.tau = tau
        self.beta1 = beta1 # img2txt hardness
        self.beta2 = beta2 # txt2img hardness

    def _hardness_weights(self, sim_matrix: torch.Tensor, beta: float) -> torch.Tensor:
        B = sim_matrix.size(0)
        # mask diagonal
        mask = ~torch.eye(B, dtype=torch.bool, device=sim_matrix.device)
        scaled = beta * sim_matrix / self.tau # (B, B)
        # softmax over negatives only
        neg_scaled = scaled.masked_fill(~mask, float('-inf'))
        weights = F.softmax(neg_scaled, dim=1) * (B - 1) # (B, B)
        return weights

    def forward(self, image_feat: torch.Tensor, text_feat: torch.Tensor) -> torch.Tensor:
        # forward (L2 normalised img and txt embedding (B, D) for each) --> scalar DHN-NCE loss
        B = image_feat.size(0)
        sim = image_feat @ text_feat.t() # (B, B)
        mask = ~torch.eye(B, dtype=torch.bool, device=sim.device)

        # img2txt
        w_v2t = self._hardness_weights(sim, self.beta1) # (B, B)
        pos_v2t = (image_feat * text_feat).sum(dim=1) / self.tau # (B,)
        neg_v2t = (sim / self.tau).masked_fill(~mask, float('-inf'))
        neg_v2t = neg_v2t + w_v2t.masked_fill(~mask, 0).log()
        loss_v2t = (-pos_v2t + torch.logsumexp(neg_v2t, dim=1)).mean()

        # txt2img
        w_t2v = self._hardness_weights(sim.t(), self.beta2)
        pos_t2v = (text_feat * image_feat).sum(dim=1) / self.tau
        neg_t2v = (sim.t() / self.tau).masked_fill(~mask, float('-inf'))
        neg_t2v = neg_t2v + w_t2v.masked_fill(~mask, 0).log()
        loss_t2v = (-pos_t2v + torch.logsumexp(neg_t2v, dim=1)).mean()

        return loss_v2t + loss_t2v

Soft Patch-level Contrastive Loss

In [7]:
class SoftPatchContrastiveLoss(nn.Module):
    def __init__(self, tau: float = 0.2):
        super().__init__()
        self.tau = tau

    def _soft_ce(self, logits: torch.Tensor, soft_targets: torch.Tensor) -> torch.Tensor:
        log_probs = F.log_softmax(logits, dim=1)
        return -(soft_targets * log_probs).sum(dim=1).mean()

    def forward(self, patch_vis_feat: torch.Tensor, text_feat: torch.Tensor) -> torch.Tensor:
        # L2 normalise
        pv = F.normalize(patch_vis_feat, dim=-1)
        pt = F.normalize(text_feat, dim=-1)

        # soft targets from text-text similarity
        G = F.softmax(pt @ pt.t() / self.tau, dim=1) # (B, B)

        # cross-modal logits
        logits_t2v = pt @ pv.t()
        logits_v2t = pv @ pt.t()

        loss = 0.5 * (self._soft_ce(logits_t2v, G) + self._soft_ce(logits_v2t, G.t()))
        return loss

Segmentation Loss  (Dice + BCE)

In [8]:
class SegmentationLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def dice_loss(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred   = torch.sigmoid(pred).view(-1)
        target = target.view(-1).float()
        inter  = (pred * target).sum()
        return 1 - (2 * inter + self.smooth) / (pred.sum() + target.sum() + self.smooth)

    def bce_loss(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return F.binary_cross_entropy_with_logits(pred, target.float())

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return 0.5 * self.dice_loss(pred, target) + 0.5 * self.bce_loss(pred, target)

PVL Adapter

In [9]:
class ProbabilisticCrossAttention(nn.Module):
    def __init__(self, d_s: int, d_a: int, beta: float = 2.35):
        super().__init__()
        self.d_a  = d_a
        self.beta = beta

        self.W_Q = nn.Linear(d_s, d_a, bias=False)
        self.W_K = nn.Linear(d_s, 2 * d_a, bias=False) # [K_mu | K_logvar]
        self.W_V = nn.Linear(d_s, 2 * d_a, bias=False) # [V_mu | V_logvar]
        self.W_O = nn.Linear(d_a, d_s, bias=False)

        # residual gate initialised to sigmoid(0) = 0.5
        self.gate = nn.Parameter(torch.zeros(1))

    def forward(self, query_seq: torch.Tensor, context_seq: torch.Tensor, training: bool = True, n_samples: int = 1) -> torch.Tensor:
        Q = self.W_Q(query_seq)

        KV = self.W_K(context_seq)
        K_mu, K_logvar = KV.chunk(2, dim=-1)

        VV = self.W_V(context_seq)
        V_mu, V_logvar = VV.chunk(2, dim=-1)

        # variances via softplus (numerically stable vs exp)
        K_var = F.softplus(K_logvar)
        V_var = F.softplus(V_logvar)

        # confidence-weighted attention
        # S_mu = Q @ K_mu^T / sqrt(Da)
        S_mu = torch.bmm(Q, K_mu.transpose(1, 2)) / math.sqrt(self.d_a)

        # S_sigma = (Q^2) @ K_var^T / Da
        Q2 = Q ** 2
        S_sig = torch.bmm(Q2, K_var.transpose(1, 2)) / self.d_a 
        S_sig = S_sig.sqrt().clamp(min=1e-6)

        # A = softmax(S_mu - beta * S_sigma)
        A = F.softmax(S_mu - self.beta * S_sig, dim=-1)

        # value sampling
        if training or n_samples > 1:
            eps = torch.randn_like(V_mu)
            V_sample = V_mu + eps * V_var.sqrt()
        else:
            V_sample = V_mu

        O = torch.bmm(A, V_sample)
        O_proj = self.W_O(O)

        # residual gate
        g = torch.sigmoid(self.gate)
        return g * O_proj + (1 - g) * query_seq

In [10]:
class PVLAdapter(nn.Module):
    def __init__(self, d_v: int, d_t: int, d_s: int = 256, d_a: int = 64, beta: float = 2.35):
        # d_v: vision token dimension
        # d_t: text token dimension
        # d_s: shared lower dimensional bottleneck
        # d_a: attention head dimension inside AttnPVL
        # beta: confidence penalty scaling
        
        super().__init__()
        # down projections
        self.down_v = nn.Linear(d_v, d_s, bias=False)
        self.down_t = nn.Linear(d_t, d_s, bias=False)

        # bidirectional probabilistic attention
        self.attn_v2t = ProbabilisticCrossAttention(d_s, d_a, beta)
        self.attn_t2v = ProbabilisticCrossAttention(d_s, d_a, beta)

        # up projections (back to original dims)
        self.up_v = nn.Linear(d_s, d_v, bias=False)
        self.up_t = nn.Linear(d_s, d_t, bias=False)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, vis_tokens: torch.Tensor, txt_tokens: torch.Tensor, training: bool = True) -> Tuple[torch.Tensor, torch.Tensor]:
        v = self.down_v(vis_tokens) # (B, Tv, Ds)
        t = self.down_t(txt_tokens) # (B, Tt, Ds)

        # vision attends to text then text attends to updated vision
        v_prime = self.attn_v2t(v, t, training)
        t_prime = self.attn_t2v(t, v_prime, training)

        # up-project and add residual
        vis_out = vis_tokens + self.up_v(v_prime)
        txt_out = txt_tokens + self.up_t(t_prime)

        return vis_out, txt_out # updated (vis_tokens, txt_tokens) with residual connections

Segmentation Head

In [11]:
class SegmentationHead(nn.Module):
    def __init__(self, d_model: int, patch_grid: int = 14, n_upscale: int = 2):
        # d_model: CLIP embedding dimension
        # patch_grid: sqrt(nbr of patches: 14 for ViT-B/16 on 224px)
        # n_upscale: nbr of 2x upscale + conv blocks
        
        super().__init__()
        self.patch_grid = patch_grid

        # text MLP to map [EOS] -> compatible embedding
        self.text_mlp = nn.Sequential(nn.Linear(d_model, d_model), nn.GELU(), nn.Linear(d_model, d_model))

        # upscale blocks (each doubles spatial resolution)
        layers = []
        for _ in range(n_upscale):
            layers += [
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(1, 1, kernel_size=3, padding=1),
                nn.GELU(),
            ]
        self.upscale = nn.Sequential(*layers)

    def forward(self, patch_tokens: torch.Tensor, text_cls: torch.Tensor) -> torch.Tensor:
        B, P, D = patch_tokens.shape
        G = self.patch_grid
        t = F.normalize(self.text_mlp(text_cls), dim=-1) # (B, D)
        v = F.normalize(patch_tokens, dim=-1) # (B, P, D)

        # dot product similarity: (B, P)
        sim = (v * t.unsqueeze(1)).sum(dim=-1) # (B, P)

        # reshape to spatial grid
        logits_map = sim.view(B, 1, G, G) # (B, 1, G, G)

        # upscale to higher resolution
        logits_map = self.upscale(logits_map) # (B, 1, H, W)
        return logits_map # raw segmentation logits of shape (B, 1, H_out, W_out)

CLIP Encoder Wrappers with Interleaved PVL Adapters

In [12]:
class CLIPEncoderWithPVL(nn.Module):
    # wraps huggingFace style CLIP visual/text encoder so PVL will be called after every transformer encoder layer
    # adapter list has len(encoder.layers) entries: one per layer

    def __init__(
        self,
        clip_encoder: nn.Module, # frozen pre-trained CLIP encoder (ViT or text)
        adapters: nn.ModuleList, # one PVLAdapter per encoder layer
        is_vision: bool = True,
    ):
        super().__init__()
        self.encoder   = clip_encoder
        self.adapters  = adapters
        self.is_vision = is_vision

    def forward_vision(self, hidden_states: torch.Tensor, other_tokens: torch.Tensor, training: bool = True) -> Tuple[torch.Tensor, torch.Tensor]:
        # iterate over ViT layers: apply PVL after each layer
        
        # standard huggingFace CLIP ViT layers
        for i, layer in enumerate(self.encoder.layers):
            hidden_states = layer(hidden_states, attention_mask=None, causal_attention_mask=None)[0]
            hidden_states, other_tokens = self.adapters[i](hidden_states, other_tokens, training)
        return hidden_states, other_tokens # final_hidden_states, updated_other_tokens

    def forward_text(self, hidden_states: torch.Tensor, attention_mask: Optional[torch.Tensor], causal_mask: Optional[torch.Tensor], other_tokens: torch.Tensor, training: bool = True) -> Tuple[torch.Tensor, torch.Tensor]:
        # iterate onver text layers: apply PVL after each layer
        for i, layer in enumerate(self.encoder.layers):
            hidden_states = layer(
                hidden_states,
                attention_mask=attention_mask,
                causal_attention_mask=causal_mask,
            )[0]
            # adapters[i] adapts text tokens (query) against vision tokens (context)
            other_tokens, hidden_states = self.adapters[i](other_tokens, hidden_states, training)
        return hidden_states, other_tokens # final_hidden_states, updated_other_tokens

Full Hybrid Model

In [13]:
class HybridMedCLIPSeg(nn.Module):
    def __init__(
        self,
        clip_model, # open_clip/huggingFace CLIP model
        d_v: int = 768, # ViT hidden dim (ViT-B = 768)
        d_t: int = 768, # text hidden dim
        d_s: int = 256, # PVL bottleneck dim
        d_a: int = 64, # PVL attention head dim
        beta: float = 2.35, # PVL confidence penalty
        patch_grid: int = 14, # sqrt(n_patches) for ViT-B/16 on 224px
        n_upscale: int = 2, # nbr of 2x upscale blocks in seg head
        n_vis_layers: int = 12, # nbr of ViT transformer layers
        n_txt_layers: int = 12, # nbr of text transformer layers
    ):
        super().__init__()
        self.clip = clip_model

        # freeze CLIP
        for param in self.clip.parameters():
            param.requires_grad_(False)

        # PVL adapters (one per ViT/text layer)
        # vision-side adapters: vision tokens are queries, text tokens are context
        self.vis_adapters = nn.ModuleList([PVLAdapter(d_v, d_t, d_s, d_a, beta) for _ in range(n_vis_layers)])

        # text-side adapters: text tokens are queries, vision tokens are context
        self.txt_adapters = nn.ModuleList([PVLAdapter(d_t, d_v, d_s, d_a, beta) for _ in range(n_txt_layers)])

        # segmentation head
        self.seg_head = SegmentationHead(d_v, patch_grid, n_upscale)
        self.patch_grid = patch_grid
        self.n_vis_layers = n_vis_layers
        self.n_txt_layers = n_txt_layers

    # extract layer lists from different CLIP APIs
    def _get_vis_layers(self):
        enc = self.clip.visual
        if hasattr(enc, 'transformer'): # open_clip ViT
            return enc.transformer.resblocks
        if hasattr(enc, 'encoder'): # huggingFace CLIP vision model
            return enc.encoder.layers
        raise AttributeError("can't find ViT layer list in clip.visual")

    def _get_txt_layers(self):
        if hasattr(self.clip, 'transformer'): # open_clip
            return self.clip.transformer.resblocks
        if hasattr(self.clip, 'text_model'): # huggingFace CLIP
            return self.clip.text_model.encoder.layers
        raise AttributeError("can't find text layer list in clip")

    # Open-CLIP style forward
    def _encode_vision_openclip(self, pixel_values: torch.Tensor, txt_tokens: torch.Tensor, training: bool) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        enc = self.clip.visual
        # patch embedding + positional embedding (frozen)
        with torch.no_grad():
            x = enc.conv1(pixel_values) # (B, D, G, G)
            x = x.reshape(x.shape[0], x.shape[1], -1) # (B, D, P)
            x = x.permute(0, 2, 1) # (B, P, D)
            
            # prepend CLS token
            cls = enc.class_embedding.unsqueeze(0).unsqueeze(0)
            cls = cls.expand(x.shape[0], -1, -1)
            x = torch.cat([cls, x], dim=1) # (B, P+1, D)
            x = x + enc.positional_embedding.unsqueeze(0)
            x = enc.ln_pre(x)

        # layer-by-layer with PVL adapters
        vis_layers = self._get_vis_layers()
        for i, layer in enumerate(vis_layers):
            with torch.no_grad():
                x = layer(x)
            # PVL adapter: vis is query, txt is context
            x, txt_tokens = self.vis_adapters[i](x, txt_tokens, training)

        with torch.no_grad():
            x = enc.ln_post(x)

        patch_tokens = x[:, 1:, :] # (B, P, D)
        cls_token = x[:, 0, :] # (B, D)
        return patch_tokens, cls_token, txt_tokens

    def _encode_text_openclip(self, input_ids: torch.Tensor, vis_tokens: torch.Tensor, training: bool) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        enc = self.clip
        with torch.no_grad():
            x = enc.token_embedding(input_ids)
            x = x + enc.positional_embedding[:x.size(1)]
            x = x.permute(1, 0, 2) # NLD -> LND (open_clip convention)

        txt_layers = self._get_txt_layers()
        attn_mask  = enc.build_attention_mask().to(x.device) if hasattr(enc, 'build_attention_mask') else None

        for i, layer in enumerate(txt_layers):
            with torch.no_grad():
                x = layer(x, attn_mask=attn_mask)
            # x is (L, B, D) in open_clip convention
            x_bld = x.permute(1, 0, 2) # (B, L, D)
            x_bld, vis_tokens = self.txt_adapters[i](x_bld, vis_tokens, training)
            x = x_bld.permute(1, 0, 2) # back to (L, B, D)

        x = x.permute(1, 0, 2) # (B, L, D)
        with torch.no_grad():
            x = enc.ln_final(x)

        # EOS position: argmax of input_ids
        eos_pos = input_ids.argmax(dim=-1)
        eos_token = x[torch.arange(x.size(0)), eos_pos] # (B, D)
        return x, eos_token, vis_tokens

    # huggingFace CLIP style forward
    def _encode_vision_hf(self, pixel_values: torch.Tensor, txt_tokens: torch.Tensor, training: bool) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        enc = self.clip.vision_model
        with torch.no_grad():
            x = enc.embeddings(pixel_values)
            x = enc.pre_layrnorm(x) if hasattr(enc, 'pre_layrnorm') else x

        vis_layers = enc.encoder.layers
        for i, layer in enumerate(vis_layers):
            with torch.no_grad():
                x = layer(x, attention_mask=None, causal_attention_mask=None)[0]
            x, txt_tokens = self.vis_adapters[i](x, txt_tokens, training)

        with torch.no_grad():
            x = enc.post_layernorm(x)

        patch_tokens = x[:, 1:, :]
        cls_token = x[:, 0, :]
        return patch_tokens, cls_token, txt_tokens

    def _encode_text_hf(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor], vis_tokens: torch.Tensor, training: bool) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        enc = self.clip.text_model
        with torch.no_grad():
            x = enc.embeddings(input_ids=input_ids, position_ids=None)

        causal_mask = enc._build_causal_attention_mask(
            x.size(0), x.size(1), x.dtype
        ).to(x.device) if hasattr(enc, '_build_causal_attention_mask') else None

        for i, layer in enumerate(enc.encoder.layers):
            with torch.no_grad():
                x = layer(x, attention_mask=attention_mask, causal_attention_mask=causal_mask)[0]
            x, vis_tokens = self.txt_adapters[i](x, vis_tokens, training)

        with torch.no_grad():
            x = enc.final_layer_norm(x)

        # EOS: position of highest input_id value (CLIP convention)
        eos_pos = input_ids.argmax(dim=-1)
        eos_token = x[torch.arange(x.size(0)), eos_pos]
        return x, eos_token, vis_tokens

    # unified forward
    def forward(self, pixel_values: torch.Tensor, input_ids: torch.Tensor,
                attention_mask: Optional[torch.Tensor] = None, 
                output_size: Optional[Tuple[int, int]] = None,
                training: bool = True, clip_api: str = 'openclip') -> dict:
        # returns dict with:
        # seg_logits: (B, 1, H_out, W_out) raw segmentation logits
        # image_feat: (B, D) global image features (L2-normed)
        # text_feat: (B, D) global text  features (L2-normed)
        # patch_tokens': (B, P, D) image patch tokens (for soft contrastive)

        B = pixel_values.size(0)

        # initial text embedding (needed as context for first vis layer)
        if clip_api == 'openclip':
            with torch.no_grad():
                txt_init = self.clip.token_embedding(input_ids)
                txt_init = txt_init + self.clip.positional_embedding[:txt_init.size(1)]
                # (B, L, D): will serve as initial text token sequence
        else:
            with torch.no_grad():
                txt_init = self.clip.text_model.embeddings(input_ids=input_ids)

        # encode vision with interleaved PVL (vision uses text as context)
        if clip_api == 'openclip':
            patch_tokens, cls_token, txt_tokens = self._encode_vision_openclip(pixel_values, txt_init, training)
            # encode text with interleaved PVL (text uses vision as context)
            _, eos_token, _ = self._encode_text_openclip(input_ids, patch_tokens, training)
        else:
            patch_tokens, cls_token, txt_tokens = self._encode_vision_hf(pixel_values, txt_init, training)
            _, eos_token, _ = self._encode_text_hf(input_ids, attention_mask, patch_tokens, training)

        # project to shared CLIP space (open_clip has projection layers)
        with torch.no_grad():
            if clip_api == 'openclip' and hasattr(self.clip.visual, 'proj'):
                image_feat = cls_token @ self.clip.visual.proj
            else:
                image_feat = cls_token
            if clip_api == 'openclip' and hasattr(self.clip, 'text_projection'):
                text_feat = eos_token @ self.clip.text_projection
            else:
                text_feat = eos_token

        image_feat = F.normalize(image_feat, dim=-1)
        text_feat  = F.normalize(text_feat,  dim=-1)

        # average-pooled patch tokens for soft contrastive loss
        avg_patch = patch_tokens.mean(dim=1) # (B, D)

        # segmentation logits
        seg_logits = self.seg_head(patch_tokens, eos_token) # (B, 1, H, W)

        if output_size is not None:
            seg_logits = F.interpolate(seg_logits, size=output_size, mode='bilinear', align_corners=False)

        return {
            'seg_logits':   seg_logits,
            'image_feat':   image_feat,
            'text_feat':    text_feat,
            'patch_tokens': patch_tokens,
            'avg_patch':    avg_patch,
        }

    def predict_with_uncertainty(
        self,
        pixel_values: torch.Tensor,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        output_size: Optional[Tuple[int, int]] = None,
        n_samples: int = 30,
        clip_api: str = 'openclip',
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Monte Carlo sampling over PVL value distributions, it returns (mean_pred, uncertainty_map) both in [0,1]
        self.eval()
        preds = []
        with torch.no_grad():
            for _ in range(n_samples):
                out = self.forward(
                    pixel_values, input_ids, attention_mask,
                    output_size=output_size, training=True, # training=True enables sampling
                    clip_api=clip_api,
                )
                preds.append(torch.sigmoid(out['seg_logits']))

        preds = torch.stack(preds, dim=0) # (S, B, 1, H, W)
        mean_pred  = preds.mean(dim=0) # (B, 1, H, W)

        # entropy-based uncertainty
        p = mean_pred.clamp(1e-6, 1 - 1e-6)
        uncertainty = -(p * p.log() + (1 - p) * (1 - p).log())  # (B, 1, H, W)

        return mean_pred, uncertainty

Combined Training Loss

In [14]:
class HybridLoss(nn.Module):
    def __init__(
        self,
        lambda_seg: float = 0.5,
        lambda_dhn: float = 1.0,
        lambda_scon: float = 0.1,
        tau_dhn: float = 0.6,
        beta1: float = 0.15,
        beta2: float = 0.15,
        tau_scon: float = 0.2,
    ):
        super().__init__()
        self.lambda_seg = lambda_seg
        self.lambda_dhn = lambda_dhn
        self.lambda_scon = lambda_scon

        self.seg_loss = SegmentationLoss()
        self.dhn_loss = DHNNCELoss(tau_dhn, beta1, beta2)
        self.scon_loss = SoftPatchContrastiveLoss(tau_scon)

    def forward(
        self,
        seg_logits: torch.Tensor, # (B, 1, H, W)
        masks: torch.Tensor, # (B, 1, H, W) binary ground-truth
        image_feat: torch.Tensor, # (B, D) L2-normed
        text_feat: torch.Tensor, # (B, D) L2-normed
        avg_patch: torch.Tensor, # (B, D) avg patch tokens
    ) -> Tuple[torch.Tensor, dict]:

        l_seg = self.seg_loss(seg_logits, masks)
        l_dhn = self.dhn_loss(image_feat, text_feat)
        l_scon = self.scon_loss(avg_patch, text_feat)

        total = (self.lambda_seg  * l_seg + self.lambda_dhn  * l_dhn + self.lambda_scon * l_scon)

        return total, {
            'total': total.item(),
            'seg': l_seg.item(),
            'dhn': l_dhn.item(),
            'scon': l_scon.item(),
        }

Dataset

In [15]:
class MRISegDataset(Dataset):
    IMAGE_EXT = ('.nii', '.nii.gz', '.png', '.jpg', '.jpeg')

    def __init__(self, root: str, split: str = 'train', image_size: int = 224, max_text_len: int = 77, tokenizer=None, clip_api: str = 'openclip'):
        self.root = Path(root) / split
        self.image_size = image_size
        self.max_text_len = max_text_len
        self.tokenizer = tokenizer
        self.clip_api = clip_api
        self.samples: List[Dict] = []
        self._build_index()

    def _build_index(self):
        img_dir = self.root / 'images'
        mask_dir = self.root / 'masks'
        report_dir = self.root / 'reports'

        if not img_dir.exists():
            log.warning(f"image dir is not found: {img_dir}")
            return

        for img_path in sorted(img_dir.iterdir()):
            if img_path.suffix not in ('.nii', '.gz', '.png', '.jpg', '.jpeg'):
                continue

            stem = img_path.name.replace('.nii.gz', '').replace('.nii', '')
            stem = Path(stem).stem

            mask_path = self._find_file(mask_dir,   stem)
            report_path = self._find_file(report_dir, stem, ext='.txt')

            if mask_path is None:
                log.debug(f"no mask found for {stem}, skipping")
                continue

            text = self._load_text(report_path)

            if str(img_path).endswith(('.nii', '.nii.gz')) and HAS_NIBABEL:
                slices = self._nifti_slices(img_path, mask_path)
            else:
                slices = [{'img_path': str(img_path), 'msk_path': str(mask_path), 'slice_idx': None}]

            for s in slices:
                s['text'] = text
                self.samples.append(s)

        log.info(f"loaded {len(self.samples)} slices from {self.root}")

    def _find_file(self, directory: Path, stem: str, ext: str = '') -> Optional[Path]:
        if not directory.exists():
            return None
        patterns = [stem + ext, stem + '.nii.gz', stem + '.nii', stem + '.png', stem + '.jpg']
        for p in patterns:
            f = directory / p
            if f.exists():
                return f
        # fuzzy: any file starting with stem
        for f in directory.iterdir():
            if f.stem.startswith(stem):
                return f
        return None

    @staticmethod
    def _load_text(path: Optional[Path]) -> str:
        if path is None or not path.exists():
            return "MRI scan showing anatomical structures"
        return path.read_text(encoding='utf-8').strip()

    @staticmethod
    def _nifti_slices(img_path: Path, msk_path: Path) -> List[Dict]:
        slices = []
        img_nii = nib.load(str(img_path))
        n_slices = img_nii.shape[-1] if img_nii.ndim == 3 else 1
        for i in range(n_slices):
            slices.append({'img_path': str(img_path), 'msk_path': str(msk_path), 'slice_idx': i})
        return slices

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict:
        s = self.samples[idx]
        mask  = self._load_mask( s['msk_path'], s['slice_idx'])
        text  = s['text']

        # tokenise
        if self.clip_api == 'openclip' and HAS_OPENCLIP:
            tokens = open_clip.tokenize([text], context_length=self.max_text_len)[0]
            input_ids  = tokens
            attn_mask  = (tokens != 0).long()
        elif HAS_HF and self.tokenizer is not None:
            enc = self.tokenizer(text, max_length=self.max_text_len, padding='max_length', truncation=True, return_tensors='pt')
            input_ids = enc['input_ids'].squeeze(0)
            attn_mask = enc['attention_mask'].squeeze(0)
        else:
            # dummy tokens for testing without real models
            input_ids = torch.zeros(self.max_text_len, dtype=torch.long)
            attn_mask = torch.zeros(self.max_text_len, dtype=torch.long)

        return {
            'pixel_values': image, # (3, H, W) float32
            'input_ids': input_ids, # (L,) long
            'attention_mask': attn_mask,
            'mask': mask, # (1, H, W) float32 binary
            'text': text,
        }

    def _load_image(self, path: str, slice_idx: Optional[int]) -> torch.Tensor:
        if path.endswith(('.nii', '.nii.gz')) and HAS_NIBABEL:
            vol = nib.load(path).get_fdata()
            img = vol[..., slice_idx] if slice_idx is not None else vol
            img = self._normalise_array(img)
            img = Image.fromarray((img * 255).astype(np.uint8)).convert('RGB')
        else:
            img = Image.open(path).convert('RGB')

        img = img.resize((self.image_size, self.image_size), Image.BILINEAR)
        tensor = torch.from_numpy(np.array(img)).float() / 255.0
        tensor = tensor.permute(2, 0, 1) # (3, H, W)

        # normalise with CLIP mean/std
        mean = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
        std  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)
        return (tensor - mean) / std

    def _load_mask(self, path: str, slice_idx: Optional[int]) -> torch.Tensor:
        if path.endswith(('.nii', '.nii.gz')) and HAS_NIBABEL:
            vol = nib.load(path).get_fdata()
            msk = vol[..., slice_idx] if slice_idx is not None else vol
            msk = (msk > 0).astype(np.float32)
            msk = Image.fromarray((msk * 255).astype(np.uint8)).convert('L')
        else:
            msk = Image.open(path).convert('L')

        msk = msk.resize((self.image_size, self.image_size), Image.NEAREST)
        tensor = torch.from_numpy(np.array(msk)).float() / 255.0
        return (tensor > 0.5).float().unsqueeze(0) # (1, H, W)

    @staticmethod
    def _normalise_array(arr: np.ndarray) -> np.ndarray:
        mn, mx = arr.min(), arr.max()
        if mx - mn < 1e-8:
            return np.zeros_like(arr, dtype=np.float32)
        return ((arr - mn) / (mx - mn)).astype(np.float32)

Metrics

In [16]:
class SegmentationMetrics:
    # IoU, Dice, Precision, Recall, F1, AUC (pixel-level), Normalised Surface Distance (NSD) over epoch

    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold
        self.reset()

    def reset(self):
        self._tp = self._fp = self._fn = self._tn = 0.0
        self._auc_preds:  List[torch.Tensor] = []
        self._auc_labels: List[torch.Tensor] = []

    def update(self, preds: torch.Tensor, labels: torch.Tensor):
        # preds: (B, 1, H, W) proba maps in [0,1]
        # labels: (B, 1, H, W) binary ground-truth

        preds  = preds.detach().cpu().float()
        labels = labels.detach().cpu().float()
        self._tp += (binary * labels).sum().item()
        self._fp += (binary * (1 - labels)).sum().item()
        self._fn += ((1 - binary) * labels).sum().item()
        self._tn += ((1 - binary) * (1 - labels)).sum().item()

        self._auc_preds.append(preds.view(-1))
        self._auc_labels.append(labels.view(-1))

    def compute(self) -> Dict[str, float]:
        tp, fp, fn, tn = self._tp, self._fp, self._fn, self._tn
        smooth = 1e-6

        precision = tp / (tp + fp + smooth)
        recall = tp / (tp + fn + smooth)
        f1 = 2 * precision * recall / (precision + recall + smooth)
        iou = tp / (tp + fp + fn + smooth)
        dice = 2 * tp / (2 * tp + fp + fn + smooth)
        accuracy  = (tp + tn) / (tp + tn + fp + fn + smooth)

        # AUC via trapezoidal rule
        auc = self._compute_auc()

        return {
            'IoU': round(iou * 100, 2),
            'Dice': round(dice * 100, 2),
            'Precision': round(precision * 100, 2),
            'Recall': round(recall * 100, 2),
            'F1': round(f1 * 100, 2),
            'Accuracy': round(accuracy * 100, 2),
            'AUC': round(auc * 100, 2),
        }

    def _compute_auc(self) -> float:
        if not self._auc_preds:
            return 0.0
        preds  = torch.cat(self._auc_preds).numpy()
        labels = torch.cat(self._auc_labels).numpy().astype(int)

        # sub-sample for speed on large arrays
        if len(preds) > 100_000:
            idx = np.random.choice(len(preds), 100_000, replace=False)
            preds, labels = preds[idx], labels[idx]

        # sort by descending score
        order  = np.argsort(-preds)
        labels = labels[order]

        n_pos = labels.sum()
        n_neg = len(labels) - n_pos
        if n_pos == 0 or n_neg == 0:
            return 0.5

        tpr = np.cumsum(labels)     / n_pos
        fpr = np.cumsum(1 - labels) / n_neg
        tpr = np.concatenate([[0], tpr])
        fpr = np.concatenate([[0], fpr])
        return float(np.trapz(tpr, fpr))

In [17]:
def compute_nsd(pred: np.ndarray, gt: np.ndarray, tau: float = 2.0) -> float:
    # Normalised Surface Distance (NSD) using border pixels and tau is tolerance in pixels
    
    try:
        from scipy.ndimage import binary_erosion, distance_transform_edt
    except ImportError:
        return 0.0

    def surface(mask):
        return mask & ~binary_erosion(mask)

    pred_surf = surface(pred.astype(bool))
    gt_surf = surface(gt.astype(bool))

    if not pred_surf.any() or not gt_surf.any():
        return 0.0

    dist_pred = distance_transform_edt(~pred_surf)
    dist_gt = distance_transform_edt(~gt_surf)

    n_correct = (dist_gt[pred_surf] <= tau).sum() + (dist_pred[gt_surf] <= tau).sum()
    n_total = pred_surf.sum() + gt_surf.sum()
    return float(n_correct / (n_total + 1e-8))

Model Factory

In [18]:
def build_model(cfg: dict, device: torch.device) -> Tuple[HybridMedCLIPSeg, object]:
    clip_api = cfg.get('clip_api', 'openclip')
    pretrained = cfg.get('pretrained', 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')

    if clip_api == 'openclip':
        if not HAS_OPENCLIP:
            raise ImportError("open_clip is required")
        clip_model, _, _ = open_clip.create_model_and_transforms(cfg.get('arch', 'ViT-B-16'), pretrained=pretrained)
        tokenizer = open_clip.get_tokenizer(cfg.get('arch', 'ViT-B-16'))
    else:
        if not HAS_HF:
            raise ImportError("transformers is required")
        clip_model = CLIPModel.from_pretrained(pretrained)
        tokenizer  = AutoTokenizer.from_pretrained(pretrained)

    model = HybridMedCLIPSeg(
        clip_model = clip_model,
        d_v = cfg.get('d_v', 768),
        d_t = cfg.get('d_t', 768),
        d_s = cfg.get('d_s', 256),
        d_a = cfg.get('d_a', 64),
        beta = cfg.get('beta', 2.35),
        patch_grid = cfg.get('patch_grid', 14),
        n_upscale = cfg.get('n_upscale', 2),
        n_vis_layers = cfg.get('n_vis_layers', 12),
        n_txt_layers = cfg.get('n_txt_layers', 12),
    ).to(device)

    return model, tokenizer

Trainer

In [19]:
class Trainer:
    def __init__(self, model: HybridMedCLIPSeg, cfg: dict, device: torch.device):
        self.model = model
        self.cfg = cfg
        self.device = device

        self.criterion = HybridLoss(
            lambda_seg = cfg.get('lambda_seg', 0.5),
            lambda_dhn = cfg.get('lambda_dhn', 1.0),
            lambda_scon = cfg.get('lambda_scon', 0.1),
            tau_dhn = cfg.get('tau_dhn', 0.6),
            beta1 = cfg.get('beta1', 0.15),
            beta2 = cfg.get('beta2', 0.15),
            tau_scon = cfg.get('tau_scon', 0.2),
        )

        # only train adapters + segmentation head
        trainable = [p for p in model.parameters() if p.requires_grad]
        log.info(f"trainable params: {sum(p.numel() for p in trainable):,}")

        self.optimizer = AdamW(trainable, lr=cfg.get('lr', 3e-4), weight_decay=cfg.get('weight_decay', 1e-4))
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=cfg.get('epochs', 100), eta_min=cfg.get('lr_min', 1e-6))

        self.clip_api = cfg.get('clip_api', 'openclip')
        self.output_dir = Path(cfg.get('output_dir', './checkpoints'))
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.best_dice = 0.0
        self.history: List[Dict] = []

    def _step(self, batch: Dict, training: bool) -> Tuple[torch.Tensor, Dict, torch.Tensor]:
        pv = batch['pixel_values'].to(self.device)
        ids = batch['input_ids'].to(self.device)
        att = batch['attention_mask'].to(self.device)
        msk = batch['mask'].to(self.device)

        out = self.model(
            pixel_values = pv,
            input_ids = ids,
            attention_mask = att,
            output_size = (msk.shape[-2], msk.shape[-1]),
            training = training,
            clip_api = self.clip_api,
        )

        loss, loss_dict = self.criterion(
            seg_logits = out['seg_logits'],
            masks = msk,
            image_feat = out['image_feat'],
            text_feat = out['text_feat'],
            avg_patch = out['avg_patch'],
        )

        preds = torch.sigmoid(out['seg_logits'])
        return loss, loss_dict, preds

    def train_epoch(self, loader: DataLoader, epoch: int) -> Dict:
        self.model.train()
        metrics = SegmentationMetrics()
        total_loss = {}
        n = 0

        for batch in loader:
            self.optimizer.zero_grad()
            loss, loss_dict, preds = self._step(batch, training=True)
            loss.backward()
            nn.utils.clip_grad_norm_([p for p in self.model.parameters() if p.requires_grad], 1.0)
            self.optimizer.step()

            metrics.update(preds.detach(), batch['mask'].to(self.device))
            for k, v in loss_dict.items():
                total_loss[k] = total_loss.get(k, 0.0) + v
            n += 1

        self.scheduler.step()
        avg_loss = {k: v / n for k, v in total_loss.items()}
        seg_met = metrics.compute()
        return {**avg_loss, **seg_met}

    @torch.no_grad()
    def eval_epoch(self, loader: DataLoader, use_mc: bool = False) -> Dict:
        self.model.eval()
        metrics = SegmentationMetrics()
        total_loss = {}
        n = 0

        for batch in loader:
            if use_mc:
                pv = batch['pixel_values'].to(self.device)
                ids = batch['input_ids'].to(self.device)
                att = batch['attention_mask'].to(self.device)
                preds, _ = self.model.predict_with_uncertainty(
                    pv, ids, att,
                    output_size=(batch['mask'].shape[-2], batch['mask'].shape[-1]),
                    n_samples=self.cfg.get('mc_samples', 10),
                    clip_api=self.clip_api,
                )
                # compute loss for logging (single pass)
                loss, loss_dict, _ = self._step(batch, training=False)
            else:
                loss, loss_dict, preds = self._step(batch, training=False)

            metrics.update(preds, batch['mask'].to(self.device))
            for k, v in loss_dict.items():
                total_loss[k] = total_loss.get(k, 0.0) + v
            n += 1

        avg_loss = {k: v / n for k, v in total_loss.items()}
        seg_met = metrics.compute()
        return {**avg_loss, **seg_met}

    def save_checkpoint(self, epoch: int, metrics: Dict, tag: str = ''):
        state = {
            'epoch': epoch,
            'model': {k: v for k, v in self.model.state_dict().items() if 'vis_adapters' in k or 'txt_adapters' in k or 'seg_head' in k},
            'optimizer': self.optimizer.state_dict(),
            'metrics': metrics,
        }
        fname = self.output_dir / f'checkpoint_{tag}_ep{epoch:03d}.pt'
        torch.save(state, fname)
        log.info(f"saved checkpoint: {fname}")

    def load_checkpoint(self, path: str):
        state = torch.load(path, map_location=self.device)
        # only load adapter + head weights (CLIP stays frozen)
        self.model.load_state_dict(state['model'], strict=False)
        self.optimizer.load_state_dict(state['optimizer'])
        log.info(f"loaded checkpoint from {path}")
        return state.get('epoch', 0)

    def fit(self, train_loader: DataLoader, val_loader: DataLoader):
        epochs = self.cfg.get('epochs', 100)
        val_every = self.cfg.get('val_every', 5)
        save_every = self.cfg.get('save_every', 10)

        for epoch in range(1, epochs + 1):
            t0 = time.time()
            train_met = self.train_epoch(train_loader, epoch)
            elapsed = time.time() - t0

            log.info(
                f"[train] ep {epoch:3d}/{epochs} | "
                f"loss={train_met.get('total', 0):.4f} | "
                f"dice={train_met.get('Dice', 0):.2f}% | "
                f"IoU={train_met.get('IoU', 0):.2f}% | "
                f"time={elapsed:.1f}s"
            )

            if epoch % val_every == 0:
                val_met = self.eval_epoch(val_loader)
                log.info(
                    f"[val] ep {epoch:3d}/{epochs} | "
                    f"loss={val_met.get('total', 0):.4f} | "
                    f"dice={val_met.get('Dice', 0):.2f}% | "
                    f"IoU={val_met.get('IoU', 0):.2f}%"
                )

                if val_met.get('Dice', 0) > self.best_dice:
                    self.best_dice = val_met['Dice']
                    self.save_checkpoint(epoch, val_met, tag='best')

                self.history.append({'epoch': epoch, 'train': train_met, 'val': val_met})

            if epoch % save_every == 0:
                self.save_checkpoint(epoch, train_met, tag='periodic')

        # save history
        hist_path = self.output_dir / 'history.json'
        with open(hist_path, 'w') as f:
            json.dump(self.history, f, indent=2)
        log.info(f"training complete, best val dice: {self.best_dice:.2f}%")
        log.info(f"history saved: {hist_path}")

Test / Inference

In [20]:
@torch.no_grad()
def evaluate(
    model: HybridMedCLIPSeg,
    loader: DataLoader,
    device: torch.device,
    clip_api: str = 'openclip',
    use_mc: bool = True,
    mc_samples: int = 30,
) -> Dict:
    model.eval()
    metrics = SegmentationMetrics()
    nsd_scores: List[float] = []

    for batch in loader:
        pv = batch['pixel_values'].to(device)
        ids = batch['input_ids'].to(device)
        att = batch['attention_mask'].to(device)
        msk = batch['mask'].to(device)

        if use_mc:
            preds, uncertainty = model.predict_with_uncertainty(
                pv, ids, att,
                output_size=(msk.shape[-2], msk.shape[-1]),
                n_samples=mc_samples,
                clip_api=clip_api,
            )
        else:
            out = model(pv, ids, att,
                          output_size=(msk.shape[-2], msk.shape[-1]),
                          training=False, clip_api=clip_api)
            preds = torch.sigmoid(out['seg_logits'])
        metrics.update(preds, msk)

        # NSD per sample
        pred_np = (preds.cpu().numpy() >= 0.5).astype(np.uint8)
        msk_np = msk.cpu().numpy().astype(np.uint8)
        for i in range(pred_np.shape[0]):
            nsd_scores.append(compute_nsd(pred_np[i, 0], msk_np[i, 0]))

    result = metrics.compute()
    result['NSD'] = round(np.mean(nsd_scores) * 100, 2)
    return result

CLI Entry-point

In [37]:
def parse_args():
    p = argparse.ArgumentParser(description="train/eval hybrid")
    p.add_argument('--mode', choices=['train', 'eval'], default='train')
    p.add_argument('--data_root', type=str, required=True)
    p.add_argument('--output_dir', type=str, default='./checkpoints')
    p.add_argument('--checkpoint', type=str, default=None)

    # model
    p.add_argument('--clip_api', choices=['openclip', 'huggingface'], default='huggingface')
    p.add_argument('--arch', type=str, default='ViT-B-16')
    p.add_argument('--pretrained', type=str, default='hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
    p.add_argument('--d_s', type=int, default=256, help="PVL bottleneck dim")
    p.add_argument('--d_a', type=int, default=64, help="PVL attention dim")
    p.add_argument('--n_upscale', type=int, default=2, help="seg-head upscale blocks")

    # training
    p.add_argument('--epochs', type=int, default=100)
    p.add_argument('--batch_size', type=int, default=8)
    p.add_argument('--lr', type=float, default=3e-4)
    p.add_argument('--lambda_seg', type=float, default=0.5)
    p.add_argument('--lambda_dhn', type=float, default=1.0)
    p.add_argument('--lambda_scon', type=float, default=0.1)
    p.add_argument('--mc_samples', type=int, default=30)

    # data
    p.add_argument('--image_size', type=int, default=224)
    p.add_argument('--num_workers', type=int, default=4)

    p.add_argument('--device', type=str, default='cuda' if torch.cuda.is_available() else 'cpu')
    p.add_argument('--seed', type=int, default=42)
    
    #return p.parse_args()